In [ ]:
#Load Everything
import os
import ast
import pandas as pd

BASE_DIR = os.getcwd()
DATA_DIR = os.path.normpath(os.path.join(BASE_DIR, "..", "data"))

jobs = pd.read_csv(os.path.join(DATA_DIR, "clean_jobs.csv"))
normalized = pd.read_csv(os.path.join(DATA_DIR, "normalized_skills.csv"))
tax = pd.read_csv(os.path.join(DATA_DIR, "skill_taxonomy.csv"))

print(jobs.columns.tolist())
print(normalized.columns.tolist())

['job_id', 'job_title', 'company', 'location', 'job_description', 'experience', 'education', 'salary', 'job_type', 'clean_description']
['job_id', 'job_title', 'normalized_skills']


In [2]:
def parse_skills(x):
    if isinstance(x, str):
        return ast.literal_eval(x)
    return x if isinstance(x, list) else []

normalized["normalized_skills"] = normalized["normalized_skills"].apply(parse_skills)

# Explode: one row per (job_id, skill)
long_df = normalized.explode("normalized_skills").rename(columns={"normalized_skills": "skill"})
long_df = long_df.dropna(subset=["skill"])

print(f"{len(long_df)} job-skill rows from {normalized['job_id'].nunique()} jobs")
long_df.head(10)

52962 job-skill rows from 10000 jobs


,job_id,job_title,skill
0,JOB00001,Machine Learning Engineer,Docker
0,JOB00001,Machine Learning Engineer,Machine Learning
0,JOB00001,Machine Learning Engineer,Python
0,JOB00001,Machine Learning Engineer,Scikit-learn
0,JOB00001,Machine Learning Engineer,TensorFlow
1,JOB00002,Product Analyst,Cloud Computing
1,JOB00002,Product Analyst,Excel
1,JOB00002,Product Analyst,Product Analytics
1,JOB00002,Product Analyst,Python
1,JOB00002,Product Analyst,SQL


In [3]:
# join in job metadata + skill category

# Bring in job_title, and any other useful columns (location, job_type, posted_date if present)
long_df = long_df.merge(
    jobs[["job_id", "job_title"]],  # add more columns here if your clean_jobs.csv has them (location, date, etc.)
    on="job_id", how="left", suffixes=("", "_dup")
)

# Bring in skill category from taxonomy
skill_to_category = dict(zip(tax["skill"], tax["category"]))
long_df["skill_category"] = long_df["skill"].map(skill_to_category)

long_df.head(10)

,job_id,job_title,skill,job_title_dup,skill_category
0,JOB00001,Machine Learning Engineer,Docker,Machine Learning Engineer,DevOps
1,JOB00001,Machine Learning Engineer,Machine Learning,Machine Learning Engineer,Machine Learning
2,JOB00001,Machine Learning Engineer,Python,Machine Learning Engineer,Programming
3,JOB00001,Machine Learning Engineer,Scikit-learn,Machine Learning Engineer,Machine Learning
4,JOB00001,Machine Learning Engineer,TensorFlow,Machine Learning Engineer,Machine Learning
5,JOB00002,Product Analyst,Cloud Computing,Product Analyst,Cloud
6,JOB00002,Product Analyst,Excel,Product Analyst,Data Analytics
7,JOB00002,Product Analyst,Product Analytics,Product Analyst,Data Analytics
8,JOB00002,Product Analyst,Python,Product Analyst,Programming
9,JOB00002,Product Analyst,SQL,Product Analyst,Database


In [4]:
print(jobs.columns.tolist())
# Look for something like 'posted_date', 'date_posted', 'created_at', etc.

['job_id', 'job_title', 'company', 'location', 'job_description', 'experience', 'education', 'salary', 'job_type', 'clean_description']


In [5]:
OUTPUT_PATH = os.path.join(DATA_DIR, "dashboard_data.csv")
long_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(long_df)} rows to {OUTPUT_PATH}")
print(long_df.dtypes)

Saved 52962 rows to c:\Users\Admin\Desktop\Internship Task\Job Skill extraction\Job_Skill_Extraction\data\dashboard_data.csv
job_id            str
job_title         str
skill             str
job_title_dup     str
skill_category    str
dtype: object
